In [1]:
from simple_neural_mpc.robots import Unicycle
from simple_neural_mpc.neural_modeling.dataset import UnicycleDataset
from simple_neural_mpc.neural_modeling.dataset.datamodule import Datamodule
from simple_neural_mpc.utils import project_root
import numpy as np
from simple_neural_mpc.config.neural_config import DatasetConfig

np.set_printoptions(precision=3, suppress=True)

robot = Unicycle()
dataset = None if DatasetConfig.load_data else UnicycleDataset.generate_data(robot)
datamodule = Datamodule(dataset, savedpath=f'{project_root()}/data/unicycle')

TypeError: torch._VariableFunctionsClass.from_numpy() takes no keyword arguments

In [3]:
from simple_neural_mpc.neural_modeling.learner.pinn import Pinn

data_range = datamodule.train_data.get_range()
pinn = Pinn(robot, data_range)

In [ ]:
from simple_neural_mpc.neural_modeling.learner.trainer import PinnTrainer

trainer = PinnTrainer()
trainer.fit(pinn, datamodule)

In [ ]:
import torch
import matplotlib.pyplot as plt

pinn.load_state_dict(torch.load(f'{project_root()}/simple_neural_mpc/neural_modeling/models/unicycle_60_epochs.pth'), map_location=torch.device('cpu'))

state_init = torch.Tensor([1, 0.2, 3])
vel = torch.Tensor([0.9, 0.4])
t = torch.Tensor([0.1])

states = [state_init]

for _ in range(100):

    with torch.no_grad():
        input_pair = torch.cat([states[-1], vel, t])
        state = pinn(input_pair)
        states.append(state)

state = states[0]
states_model = [state]
for _ in range(100):

    x_dot = torch.cos(states_model[-1][2])*vel[0]
    y_dot = torch.sin(states_model[-1][2])*vel[0]
    theta_dot = vel[1]

    states_model.append(
        states_model[-1] + torch.Tensor([x_dot, y_dot, theta_dot]) * t
    )

state = torch.vstack(states).cpu().numpy()
state_model = torch.vstack(states_model).cpu().numpy()
plt.plot(state[:, 0], state[:, 1], label='pinn')
plt.plot(state_model[:, 0], state_model[:, 1], label='euler')
plt.axis('equal')
plt.legend()